# Physio Support OpenEnv Training Notebook

This notebook is a runnable training entrypoint for the Physio Support OpenEnv submission.

It mirrors the Python-script pipeline in the repo and is intended for reviewer-friendly reruns in Jupyter or Colab.


## What this notebook covers

1. Install project dependencies
2. Run the SFT scaffold
3. Optionally run bootstrap SFT
4. Run Phase 6 GRPO training
5. Regenerate PNG plots for submission

The notebook intentionally calls the repo scripts directly so that the notebook and script pipeline stay aligned.


In [ ]:
import os
from pathlib import Path

ROOT = Path.cwd()
print(ROOT)


In [ ]:
# Install repo dependencies.
!python -m pip install --upgrade pip
!python -m pip install -r requirements.txt


## Optional environment variables

Set these only if you want live model/API-backed evaluation. The environment still works without them.


In [ ]:
# Example only. Uncomment and edit if needed.
# os.environ['HF_TOKEN'] = 'your_token'
# os.environ['OPENAI_API_KEY'] = 'your_token'
# os.environ['API_BASE_URL'] = 'https://router.huggingface.co/v1'
# os.environ['MODEL_NAME'] = 'katanemo/Arch-Router-1.5B:hf-inference'


## Stage 1: supervised scaffold

This creates a LoRA adapter from structured teacher examples and produces baseline artifacts.


In [ ]:
!python train_scaffold.py --base-model Qwen/Qwen2.5-0.5B-Instruct --num-train-epochs 3 --variants-per-task 8 --output-dir artifacts/training


## Stage 2: bootstrap SFT warm start

This teaches the model the structured output contract before GRPO.


In [ ]:
!python phase55_bootstrap_sft.py --base-model Qwen/Qwen2.5-0.5B-Instruct --output-dir artifacts/phase55/bootstrap_sft --variants-per-task 8


## Stage 3: Phase 6 GRPO environment training

This runs the environment-reward training loop used for the final submission story.


In [ ]:
!python phase6_train.py --base-model Qwen/Qwen2.5-0.5B-Instruct --output-dir artifacts/phase6/grpo_training --variants-per-task 8 --num-train-epochs 2 --num-generations 4 --bootstrap-adapter-path artifacts/phase55/bootstrap_sft


## Alternative: full bootstrap-plus-GRPO run in one command


In [ ]:
# Uncomment if you want the combined pipeline instead of the separate bootstrap cell above.
# !python phase6_train.py --base-model Qwen/Qwen2.5-0.5B-Instruct --output-dir artifacts/phase6/grpo_training --variants-per-task 8 --num-train-epochs 1 --num-generations 4 --bootstrap-auto


## Regenerate submission plots


In [ ]:
!python scripts/generate_submission_plots.py


## Inspect final artifacts


In [ ]:
from pathlib import Path

for path in [
    Path('artifacts/phase6/final_results/training_summary.json'),
    Path('artifacts/phase6/grpo_smoke/reward_curve.png'),
    Path('artifacts/phase6/grpo_smoke/loss_curve.png'),
    Path('artifacts/phase6/final_results/reward_comparison.png'),
    Path('artifacts/phase6/final_results/score_comparison.png'),
]:
    print(path, path.exists())
